In [6]:

import torch ## torch let's us create tensors and also provides helper functions
import torch.nn as nn ## torch.nn gives us nn.Module(), nn.Embedding() and nn.Linear()
import torch.nn.functional as F # This gives us the softmax() and argmax()
from torch.optim import Adam ## We will use the Adam optimizer, which is, essentially,
                             ## a slightly less stochastic version of stochastic gradient descent.
from torch.utils.data import TensorDataset, DataLoader ## We'll store our data in DataLoaders

import lightning as L

In [7]:
## first, we create a dictionary that maps vocabulary tokens to id numbers...
english_token_to_id = {'lets': 0,
                       'to': 1,
                       'go': 2,
                       '<EOS>': 3 ## <EOS> = end of sequence
                      }
## ...then we create a dictionary that maps the ids to tokens. This will help us interpret the output.
## We use the "map()" function to apply the "reversed()" function to each tuple (i.e. ('lets', 0)) stored
## in the token_to_id dictionary. We then use dict() to make a new dictionary from the
## reversed tuples.
english_id_to_token = dict(map(reversed, english_token_to_id.items()))

spanish_token_to_id = {'ir': 0,
                       'vamos': 1,
                       'y': 2,
                       '<EOS>': 3}
spanish_id_to_token = dict(map(reversed, spanish_token_to_id.items()))

inputs = torch.tensor([[english_token_to_id["lets"],
                        english_token_to_id["go"]],

                       [english_token_to_id["to"],
                        english_token_to_id["go"]]])

labels = torch.tensor([[spanish_token_to_id["vamos"],
                        spanish_token_to_id["<EOS>"]],

                       [spanish_token_to_id["ir"],
                        spanish_token_to_id["<EOS>"]]])

In [8]:
print(spanish_token_to_id)

{'ir': 0, 'vamos': 1, 'y': 2, '<EOS>': 3}


In [9]:
print(english_id_to_token)

{0: 'lets', 1: 'to', 2: 'go', 3: '<EOS>'}


In [11]:
dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

## Build and Train a Seq2Seq / Encoder-Decoder Model from Scratch

In [34]:
class Seq2Seq(L.LightningModule):

    def __init__(self, max_len=2):
        super().__init__()

        # max number of tokens the decoder can generate
        self.max_output_length = max_len

        # fix randomness for reproducibility
        L.seed_everything(420)

        #################################
        # ENCODER
        #################################

        # converts input token IDs → embeddings
        self.encoder_embedding = nn.Embedding(
            num_embeddings=4,   # size of input vocabulary
            embedding_dim=2     # size of each embedding vector
        )

        # processes sequence of embeddings
        self.encoder_lstm = nn.LSTM(
            input_size=2,       # embedding size
            hidden_size=2,      # hidden state size
            num_layers=2        # stacked LSTM layers
        )

        #################################
        # DECODER
        #################################

        # converts output token IDs → embeddings
        self.decoder_embedding = nn.Embedding(
            num_embeddings=4,   # size of output vocabulary
            embedding_dim=2
        )

        # generates output sequence
        self.decoder_lstm = nn.LSTM(
            input_size=2,
            hidden_size=2,
            num_layers=2
        )

        # maps LSTM output → vocabulary logits
        self.output_layer = nn.Linear(
            in_features=2,      # hidden size
            out_features=4      # vocab size
        )

        #################################
        # LOSS
        #################################
        self.criterion = nn.CrossEntropyLoss()


    def forward(self, input_tokens, target_tokens=None):

        #################################
        # ENCODING STEP
        #################################

        # convert input tokens → embeddings
        encoder_embeds = self.encoder_embedding(input_tokens)

        # run through encoder LSTM
        _, (hidden_state, cell_state) = self.encoder_lstm(encoder_embeds)

        #################################
        # DECODING STEP
        #################################

        # start decoding with <EOS> token
        current_token_id = torch.tensor([spanish_token_to_id["<EOS>"]])

        # embed first decoder input
        decoder_embeds = self.decoder_embedding(current_token_id)

        # initialize decoder with encoder's final states
        decoder_output, (hidden_state, cell_state) = self.decoder_lstm(
            decoder_embeds, (hidden_state, cell_state)
        )

        # project to vocabulary
        logits = self.output_layer(decoder_output)
        all_outputs = logits

        # get predicted token
        predicted_token_id = torch.tensor([torch.argmax(logits)])
        all_predictions = predicted_token_id

        #################################
        # GENERATE SEQUENCE
        #################################

        for t in range(1, self.max_output_length):

            if target_tokens is None:
                # inference mode (no teacher forcing)

                # stop if <EOS> predicted
                if predicted_token_id == spanish_token_to_id["<EOS>"]:
                    break

                next_input_id = predicted_token_id

            else:
                # training mode (teacher forcing)
                next_input_id = torch.tensor([target_tokens[t - 1]])

            # embed next input token
            decoder_embeds = self.decoder_embedding(next_input_id)

            # pass through LSTM
            decoder_output, (hidden_state, cell_state) = self.decoder_lstm(
                decoder_embeds, (hidden_state, cell_state)
            )

            # project to vocab
            logits = self.output_layer(decoder_output)

            # store outputs
            all_outputs = torch.cat((all_outputs, logits), dim=0)

            # greedy prediction
            predicted_token_id = torch.tensor([torch.argmax(logits)])
            all_predictions = torch.cat((all_predictions, predicted_token_id))

        return all_outputs


    def configure_optimizers(self):
        # high learning rate for tiny dataset (fast overfitting)
        return Adam(self.parameters(), lr=0.1)


    def training_step(self, batch, batch_idx):

        # unpack batch
        input_tokens, target_tokens = batch

        # forward pass
        logits = self.forward(input_tokens[0], target_tokens[0])

        # compute loss
        loss = self.criterion(logits, target_tokens[0])

        return loss

In [35]:
model = seq2seq()
outputs = model.forward(input_tokens=torch.tensor([english_token_to_id["lets"],
                                            english_token_to_id["go"]]), ## translate "lets go", we should get "vamos <EOS>"
                        target_tokens=None)

print("Translated text:")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Seed set to 420


Translated text:
	 <EOS>


In [23]:
trainer = L.Trainer(max_epochs=40, accelerator="cpu")
trainer.fit(model, train_dataloaders=dataloader)

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name              | Type             | Params | Mode  | FLOPs
-----------------------------------------------------------------------
0 | encoder_embedding | Embedding        | 8      | train | 0    
1 | encoder_lstm      | LSTM             | 96     | train | 0    
2 | decoder_embedding | Embedding        | 8      | train | 0    
3 | decoder_lstm      | LSTM             | 96     | train | 0    
4 | output_layer      | Linear           | 12     | train | 0    
5 | loss        

Epoch 39: 100%|█| 2/2 [00:00<00:00, 417.80it/s

`Trainer.fit` stopped: `max_epochs=40` reached.


Epoch 39: 100%|█| 2/2 [00:00<00:00, 268.88it/s


In [27]:
outputs = model.forward(input_tokens=torch.tensor([english_token_to_id["to"],
                                            english_token_to_id["go"]]), ## translate "lets go", we should get "vamos <EOS>"
                        target_tokens=None)

print("Translated text:")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Translated text:
	 ir
	 <EOS>


In [29]:
total_trainable_params = sum(p.numel() for p in model.parameters())
print("Total number of trainable parameters:", total_trainable_params)

Total number of trainable parameters: 220


## Saving and Loading the trained model weights

In [30]:
trainer.save_checkpoint("seq2seq_en2es_220_trained.ckpt")

`weights_only` was not set, defaulting to `False`.


In [33]:
new_model = seq2seq.load_from_checkpoint("seq2seq_en2es_220_trained.ckpt")

outputs = new_model.forward(input_tokens=torch.tensor([english_token_to_id["lets"],
                                                english_token_to_id["go"]]),
                            target_tokens=None)

print("Translated text:")
predicted_ids = torch.argmax(outputs, dim=1)
for id in predicted_ids:
    print("\t", spanish_id_to_token[id.item()])

Seed set to 420


Translated text:
	 vamos
	 <EOS>


📘 Seq2Seq Model — Complete Notes (From Your Experiment)

⸻

1. What is a Seq2Seq Model?

A Sequence-to-Sequence (Seq2Seq) model maps one sequence to another.

Examples:

* “let’s go” → “vamos”
* “to go” → “ir”
* “I eat” → “como”

So:

input sequence  →  output sequence

The model has two main parts:

1. Encoder → reads input
2. Decoder → generates output

⸻

2. High-Level Flow

Input sentence → Encoder → (hidden_state, cell_state)
                                      ↓
                                  Decoder → Output sentence

The encoder compresses the input into a context (memory):

* hidden_state
* cell_state

The decoder uses this memory to generate output one token at a time.

⸻

3. Vocabulary (Your Setup)

You used very small vocabularies.

Example:

Input vocabulary

0: "let's"
1: "go"
2: "to"
3: "<EOS>"

Output vocabulary (Spanish)

0: "vamos"
1: "ir"
2: "<EOS>"
3: "<PAD>"

Everything is converted to token IDs before entering the model.

⸻

4. Embedding Layer

What it does

nn.Embedding(num_embeddings=4, embedding_dim=2)

It converts:

token_id → vector

Example:

"go" (1) → [0.2, -0.7]

So instead of integers, the model works with dense vectors.

⸻

5. Encoder LSTM

Code

_, (hidden_state, cell_state) = self.encoder_lstm(encoder_embeds)

What LSTM returns

output, (h_n, c_n)

(A) output

* Hidden states at every time step
* Shape:

(seq_len, batch_size, hidden_size)

You ignored it using _ because you don’t need all steps.

⸻

(B) hidden_state (h_n)

* Final hidden state
* Contains summary of input sentence

Shape:

(num_layers, batch_size, hidden_size)

⸻

(C) cell_state (c_n)

* Internal memory of LSTM
* Helps retain long-term information

⸻

Key Idea

Encoder → compresses entire sentence into (hidden_state, cell_state)

⸻

6. Decoder LSTM

Initialization

decoder_output, (hidden_state, cell_state) = self.decoder_lstm(
    decoder_embeds, (hidden_state, cell_state)
)

Important:

👉 Decoder starts with encoder’s final states

So knowledge flows from encoder → decoder.

⸻

7. Why Start Decoder with <EOS>?

current_token_id = <EOS>

In your setup, <EOS> acts like a start token.

So decoding begins like:

<EOS> → predict first word

Example:

<EOS> → vamos

⸻

8. Decoder Output

At each step:

decoder_output, (hidden_state, cell_state)

What is decoder_output?

It is the representation of the current step.

Shape:

(1, batch_size, hidden_size)

It is NOT a word yet. It is just features.

⸻

9. Linear Layer → Logits

logits = self.output_layer(decoder_output)

This converts:

hidden representation → vocabulary scores

Example:

logits = [2.1, 0.3, 5.6, -1.2]

Each number corresponds to a word.

⸻

10. Where is Softmax?

You did NOT apply softmax explicitly.

Why?

Because:

nn.CrossEntropyLoss

already does:

log_softmax + negative log likelihood

So:

* During training → no softmax needed
* During prediction → argmax works directly

⸻

11. Getting Predicted Token

predicted_token_id = torch.argmax(logits)

Why no softmax?

Because:

argmax(logits) == argmax(softmax(logits))

Softmax doesn’t change which value is largest.

⸻

12. Teacher Forcing (Critical Concept)

Code

next_input_id = target_tokens[t - 1]

Why t - 1?

Because:

input at time t = correct token at time t-1
target at time t = correct token at time t

⸻

Example

Target:

[vamos, <EOS>]

Step	Input to decoder	Target
0	<EOS>	vamos
1	vamos	<EOS>

So:

target_tokens[t - 1]

is required.

⸻

What if you use t?

Then model sees the answer before predicting → learning breaks.

⸻

13. Training vs Inference

Training Mode

if target_tokens is not None:

* uses correct previous token
* faster and stable learning

⸻

Inference Mode

if target_tokens is None:

* uses its own predictions
* can accumulate errors

⸻

14. Sequence Generation Loop

for t in range(1, max_len):

At each step:

1. Take input token
2. Embed it
3. Pass through LSTM
4. Get logits
5. Predict next token
6. Repeat

⸻

Stop Condition

if predicted_token_id == <EOS>:
    break

So generation stops when <EOS> is predicted.

⸻

15. Loss Calculation

loss = CrossEntropyLoss(logits, target_tokens)

Important:

* logits → raw scores
* target_tokens → correct indices

⸻

16. Full Data Flow (Your Model)

Input tokens
   ↓
Embedding
   ↓
Encoder LSTM
   ↓
(hidden_state, cell_state)
   ↓
Decoder start (<EOS>)
   ↓
Decoder LSTM (step-by-step)
   ↓
Linear layer
   ↓
Logits
   ↓
Argmax
   ↓
Predicted tokens

⸻

17. Key Intuition Summary

* Encoder = compress sentence
* Decoder = generate sentence
* LSTM states = memory
* Linear layer = convert features → words
* Teacher forcing = use correct past words during training
* Softmax = handled internally in loss

⸻

18. Minimal Example (Your Style)

Input:  "let's go"
Output: "vamos <EOS>"

Flow:

Encoder → understands "let's go"
Decoder:
    <EOS> → predicts "vamos"
    "vamos" → predicts "<EOS>"

⸻

19. What You Built (Important Insight)

You didn’t just code a model.

You built:

* a mini translator
* with memory (LSTM)
* trained using teacher forcing
* producing sequences step-by-step

⸻

If you want, next step can be:

* visualizing tensor shapes at every line
* or upgrading this to attention (that’s where things get really interesting)

Here’s a clean continuation in the same tone and structure for Chapter 10 (Seq2Seq). I kept it consistent with your previous post but pushed the depth further.

⸻

Continuing Chapter 10 of The StatQuest Illustrated Guide to Neural Networks and AI, I explored how Sequence-to-Sequence (Seq2Seq) models work.

After understanding how word embeddings capture relationships between words, the next step is learning how models handle entire sequences, such as translating sentences.

Unlike embeddings, which represent individual words, Seq2Seq models learn how to transform one sequence into another.

⸻

Core Idea

A Seq2Seq model consists of two parts:

* Encoder → reads and compresses the input sequence
* Decoder → generates the output sequence

The key idea is:

Input sequence → Encoder → Context (memory) → Decoder → Output sequence

Instead of predicting a single word, the model predicts a sequence of words step by step.

⸻

Encoder

The encoder processes the input sentence one token at a time using an LSTM.

Example:

Input: "let's go"

Each word is:

1. Converted into an embedding
2. Passed through the LSTM

At the end, the encoder produces:

* Hidden state
* Cell state

These act as a compressed representation of the entire sentence.

Important insight:
The encoder does not store words explicitly—it stores a summary of patterns.

⸻

Decoder

The decoder generates the output sequence using the encoder’s final states.

It works step-by-step:

1. Start with a special token (<EOS> in my setup)
2. Predict the next word
3. Feed that word back into the model
4. Repeat until <EOS> is generated

Example:

<EOS> → vamos → <EOS>

So generation is sequential and autoregressive.

⸻

Training vs Inference

Training (Teacher Forcing)

During training, the model does not rely on its own predictions.

Instead, it uses the correct previous word as input:

Input at time t → correct word at time t-1
Target → correct word at time t

Example:

Target: [vamos, <EOS>]
Step 0: input = <EOS>, predict = vamos  
Step 1: input = vamos, predict = <EOS>

This makes learning faster and more stable.

⸻

Inference (Real Use)

During inference, the model uses its own predictions:

<EOS> → vamos → <EOS>

Errors can accumulate because the model no longer has access to the correct sequence.

⸻

From Hidden State to Words

At each step:

1. LSTM produces a hidden representation
2. A linear layer converts it into logits (scores for each word)
3. The highest score is selected using argmax

Important detail:

* No explicit softmax is needed during training
* CrossEntropyLoss handles it internally

⸻

My Experiment

I used a very small dataset similar to Chapter 9:

* “let’s go” → “vamos”
* “to go” → “ir”

Goal:
Understand whether the model can:

* Learn mappings between sequences
* Distinguish different meanings of the same word (“go”)

⸻

Observations

Worked well

* The model quickly memorized mappings
* It correctly generated:
    * “vamos” for “let’s go”
    * “ir” for “to go”

Limitations

* No real generalization
* Small dataset → model mostly memorizes patterns
* Cannot handle unseen combinations

⸻

Key Insight

Seq2Seq models do not “understand” language.

They learn:

patterns of sequences → patterns of sequences

Just like embeddings learn:

word usage patterns

Seq2Seq learns:

sequence transformation patterns

⸻

Connection to Chapter 9

* Embeddings (Chapter 9) → represent words
* Seq2Seq (Chapter 10) → transform sequences

Embeddings are the input layer of meaning,
Seq2Seq is the process that uses that meaning.

⸻

Important Limitation

The encoder compresses the entire sentence into a fixed-size vector.

This creates a bottleneck:

* Works for short sentences
* Struggles with long or complex ones

This limitation leads to the next major idea:
👉 Attention mechanisms

⸻

Final Understanding

* Seq2Seq models map input sequences to output sequences
* Encoder compresses information
* Decoder generates output step-by-step
* Teacher forcing stabilizes training
* Model relies entirely on learned patterns

⸻

Takeaway

* Seq2Seq extends embeddings from words → sequences
* Learning is still statistical, not semantic
* Small data → memorization
* Large data → meaningful transformations

⸻

If you continue this series, the next logical step is attention, where the model stops relying on a single fixed summary and instead learns to focus on different parts of the input dynamically.